In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import re

In [4]:
chrome_options = Options()
#chrome_options.add_argument("--headless") # Décommenter pour cacher la fenêtre
#Lancement du navigateur (Chrome) + installation même version que le naviateur si besoin
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

BASE_URL = "https://www.cdiscount.com/search/10/climatiseur+mobile.html?page="
list_products = []
page = 1

try:
    while True:
        url = f"{BASE_URL}{page}"
        driver.get(url)
        # ! Attendre que le JS est chargé le code HTML
        print("Attente du chargement des scripts")
        #http://selenium.dev/documentation/webdriver/waits/
        driver.implicitly_wait(10)


        # Extraire les données
        # Ici, on utilise les sélecteurs de Selenium, on récupère la div qui représente le produit
        products = driver.find_elements(By.CLASS_NAME, "sc-11c3ifi-8")
        print(f"\n{len(products)} produits trouvés à la page {page} :\n")
        for product in products:
            list_products.append(product.text)

        # Boucle pour afficher les résultats
        """for i in range(len(products)):
            print(f"{products[i].text}")
            print("-" * 20)"""

        # Rechercher le lien "Suivant" à partir de la page actuelle
        links = driver.find_elements(By.CLASS_NAME, "sc-fsvrbR")
        next_link = ""
        for link in links:
            if "Suivant" in link.text:
                next_link = link
                break

        page+=1
        if next_link=="":
            break
finally:
    # !!! fermer le navigateur à la fin
    driver.quit()

Attente du chargement des scripts


KeyboardInterrupt: 

In [ ]:
df  = pd.DataFrame(list_products, columns=["Produit"])
df.to_excel("produits.xlsx", index=False)

NameError: name 'list_products' is not defined

In [5]:
products = pd.read_excel("produits.xlsx")

In [ ]:
def convertir_prix(prix):
    return float(prix.replace("€", "").replace(" ", "").replace(",", "."))

rows = []

for produit in products["Produit"]:
    lignes = str(produit).split("\n")

    lignes = [ligne.strip() for ligne in lignes if ligne.strip()]

    sponsorise = "Oui" if lignes and "Sponsorisé" in lignes[0] else "Non"
    if sponsorise == "Oui":
        nom = lignes[1] if len(lignes) > 1 else ""
    else:
        nom = lignes[0] if len(lignes) > 0 else ""

    texte = "\n".join(lignes)
    prix_actuel = ""
    prix_avant_reduction = ""
    taux_reduction = ""

    if "Prix de comparaison" in texte:
        partie_reduction = texte.split("Prix de comparaison", 1)[1]
        prix_trouves = re.findall(r"\d+(?:,\d{2})?\s*€", partie_reduction)

        if len(prix_trouves) >= 2:
            prix_avant_reduction = convertir_prix(prix_trouves[0])
            prix_actuel = convertir_prix(prix_trouves[1])

            if prix_avant_reduction > 0:
                taux_reduction = round(
                    ((prix_avant_reduction - prix_actuel) / prix_avant_reduction) * 100, 2
                )

    else:
        prix_trouves = re.findall(r"\d+(?:,\d{2})?\s*€", texte)

        if len(prix_trouves) >= 1:
            prix_actuel = convertir_prix(prix_trouves[-1])

    rows.append(
        {
            "nom": nom,
            "sponsorise": sponsorise,
            "prix_actuel": prix_actuel,
            "prix_avant_reduction": prix_avant_reduction,
            "taux_reduction_%": taux_reduction,
            "texte_complet": texte,
        }
    )

df_clean = pd.DataFrame(rows)

df_clean.to_excel("produits_avec_reductions.xlsx", index=False)
print(df_clean.head())

# Analyse du taux de réduction
# Garder uniquement les produits avec un taux de réduction
df_reductions = df_clean[df_clean["taux_reduction_%"] != ""]

if not df_reductions.empty:
    df_reductions["taux_reduction_%"] = df_reductions["taux_reduction_%"].astype(float)

    taux_max = df_reductions["taux_reduction_%"].max()

    meilleurs_produits = df_reductions[
        df_reductions["taux_reduction_%"] == taux_max
    ]

    print(f"\nMeilleur taux de réduction : {taux_max} %")
    print(f"Nombre de produits concernés : {len(meilleurs_produits)}")

    print("\nProduit(s) avec le meilleur taux de réduction :")

    for index, produit in meilleurs_produits.iterrows():
        print("-" * 50)
        print(f"Nom : {produit['nom']}")
        print(f"Sponsorisé : {produit['sponsorise']}")
        print(f"Prix actuel : {produit['prix_actuel']} €")
        print(f"Prix avant réduction : {produit['prix_avant_reduction']} €")
        print(f"Taux de réduction : {produit['taux_reduction_%']} %")

else:
    print("Aucun produit avec réduction trouvé.")


                                                 nom sponsorise  prix_actuel  \
0  Climatiseurs Portables, Refroidisseur par évap...        Oui       145.99   
1     Kit fenêtre pour climatiseur ELECTROLUX EWKIT5        Oui        79.99   
2  Climatiseur Portable avec 3 Vitesses, minuteri...        Non       145.99   
3  Climatiseur Mobile Sans Évacuation,10 L, 65 W,...        Non       235.99   
4  Climatiseur Mobile Rafraîchisseur d’Air, Réser...        Non       183.99   

  prix_avant_reduction taux_reduction_%  \
0                                         
1                                         
2                                         
3                                         
4                                         

                                       texte_complet  
0  Sponsorisé?\nClimatiseurs Portables, Refroidis...  
1  Sponsorisé?\nKit fenêtre pour climatiseur ELEC...  
2  Climatiseur Portable avec 3 Vitesses, minuteri...  
3  Climatiseur Mobile Sans Évacuation,10

C:\Users\jinch\AppData\Local\Temp\ipykernel_31788\1318933658.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reductions["taux_reduction_%"] = df_reductions["taux_reduction_%"].astype(float)
